In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))


In [2]:
import networkx as nx
from disparity_filter import DisparityFilter
from noise_corrected import NoiseCorrectedFilter
from shared import generate_graph_from_adjmx_nx, show_graph, describe_network, save_filtered_graph, dataset_backbone_combinations
import numpy as np
from config import PERCENTILE, MIN_DEGREE, ALPHA, DATASET_LIST

In [3]:
def updateHT(originalH5, state_of_nodes, name):
    new_size = len(list(filter(lambda x : x, state_of_nodes)))
    newH5 = np.zeros((originalH5.shape[0], new_size))
    
    i = 0
    for (index, node_state) in enumerate(state_of_nodes):
        if node_state:
            newH5[:, i] = originalH5[:, index]
            i+=1
    
    np.save(f"../data/npy/{name}-h5.npy", newH5)
    print(f"Updated H5  with shape {newH5.shape}")
    return newH5

In [4]:
# methods = ["disp_fil", "nois_corr"]
# datasets = ["metr-la", "pems-bay"]
# data_types = ["metr-la", "pems-bay"]
# # alphas = [0.01, 0.05]
# cuts = [f"alpah_filter{str(ALPHA).replace('.', '_')}", f"percen_filter{str(PERCENTILE).replace('.', '_')}"]

# combinations = list(itertools.product( methods, cuts))
combinations = dataset_backbone_combinations()
combinations
backbone_data_names = []

In [5]:

for datat in DATASET_LIST:
    G = generate_graph_from_adjmx_nx(f"../data/pkl/{datat}-adj_mx.pkl", datat)
    # describe_network(G, "Rede Original - "+datat.upper())
    # show_graph(G, titles="Rede Original - "+datat.upper())
    H5Matriz  = np.load(f"../data/npy/{datat}-h5.npy")
    
    for comb in combinations:
        method, cut = comb
        output_name = f"{datat}-by-{method}-with-{cut}".strip()
        backbone_data_names.append(output_name)
        print(f"Processing {output_name}...")
        if method == 'disp_fil':
            df = DisparityFilter(G)
            if cut.startswith('percen'):
                filtered_graph = df.filter_by_percentile(percentile=PERCENTILE, min_degree=MIN_DEGREE)
            else:
                filtered_graph = df.filter_by_alpha(alpha=ALPHA, min_degree=MIN_DEGREE)
            
            if filtered_graph.number_of_nodes() != G.number_of_nodes():
                updateHT(H5Matriz, df.nodesToKeep, output_name)
            save_filtered_graph(G, filtered_graph, output_name, f'../data/npy/{output_name}-adj_mx.npy')
            # describe_network(filtered_graph, f"Disparity Filter - {cut} - {datat}")
            # show_graph(filtered_graph, titles=f"Disparity Filter - {cut} - {datat}")
            
        else:
            ncf = NoiseCorrectedFilter(G, undirected=True, use_p_value=False)
            if cut.startswith('percen'):
                filtered_graph = ncf.filter_by_percentile(percentile=0.30, min_degree=MIN_DEGREE)
            else:
                filtered_graph = ncf.filter_by_alpha(alpha=ALPHA, min_degree=MIN_DEGREE)
                
            if(filtered_graph.number_of_nodes() != G.number_of_nodes()):
                updateHT(H5Matriz, df.nodesToKeep, output_name)
            save_filtered_graph(G, filtered_graph,output_name, f'../data/npy/{output_name}-adj_mx.npy')
            # describe_network(filtered_graph, f"Noise Corrected Filter - {cut} - {datat}")
            # show_graph(filtered_graph, titles=f"Noise Corrected Filter - {cut} - {datat}")
            
with open("../data/npy/backbone_data_names.txt", "w") as f:
    for name in backbone_data_names:
        f.write(f"{name}\n")

Processing metr-la-by-disp_fil-with-alpah_filter0_1...
Updated H5  with shape (34272, 138)
matriz de adjacência filtrada com shape (138, 138)
Processing metr-la-by-disp_fil-with-percen_filter0_3...
Updated H5  with shape (34272, 206)
matriz de adjacência filtrada com shape (206, 206)
Processing metr-la-by-nois_corr-with-alpah_filter0_1...
matriz de adjacência filtrada com shape (207, 207)
Processing metr-la-by-nois_corr-with-percen_filter0_3...
matriz de adjacência filtrada com shape (207, 207)
Processing pems-bay-by-disp_fil-with-alpah_filter0_1...
Updated H5  with shape (52116, 53)
matriz de adjacência filtrada com shape (53, 53)
Processing pems-bay-by-disp_fil-with-percen_filter0_3...
Updated H5  with shape (52116, 319)
matriz de adjacência filtrada com shape (319, 319)
Processing pems-bay-by-nois_corr-with-alpah_filter0_1...
matriz de adjacência filtrada com shape (325, 325)
Processing pems-bay-by-nois_corr-with-percen_filter0_3...
matriz de adjacência filtrada com shape (325, 325)